In [ ]:
!pip install -q optuna

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [ ]:
# ============================================================
# NOTEBOOK DE MODELADO — Síndrome Metabólico (etiqueta corregida)
# CELDA 1: Setup
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import numpy as np
import pandas as pd
import pickle, os, warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             balanced_accuracy_score, recall_score,
                             f1_score, confusion_matrix)
import xgboost as xgb
import lightgbm as lgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

BASE_PATH = '/content/drive/MyDrive/Colab Notebooks/Paper Simulation/'
CKPT_PATH = BASE_PATH + 'checkpoints/'
RANDOM_STATE = 42

print('✓ Setup completo')
print('  XGBoost:', xgb.__version__, '| LightGBM:', lgb.__version__)

In [ ]:
BASE_PATH = '/content/drive/MyDrive/Colab Notebooks/Paper Simulation/'
CKPT_PATH = BASE_PATH + 'checkpoints/'
OUT_PATH  = BASE_PATH + 'outputs/'
import os
os.makedirs(OUT_PATH, exist_ok=True)

In [ ]:
# ============================================================
# CELDA 2: Cargar split ya generado
# ============================================================
with open(CKPT_PATH + 'split_data.pkl', 'rb') as f:
    d = pickle.load(f)

df_train = d['df_train']
df_test  = d['df_test']
predictores_A = d['predictores_A']
predictores_B = d['predictores_B']

print(f'Train: {len(df_train)} obs | Test: {len(df_test)} obs')
print(f'Config A: {len(predictores_A)} preds | Config B: {len(predictores_B)} preds')

# Identificar columnas binarias vs continuas (para imputación)
def split_bin_cont(df, cols):
    binarias, continuas = [], []
    for c in cols:
        vals = df[c].dropna().unique()
        if set(vals).issubset({0, 1}):
            binarias.append(c)
        else:
            continuas.append(c)
    return binarias, continuas

print('✓ Split cargado')

In [ ]:
# ============================================================
# CELDA 3: Funciones auxiliares
# ============================================================

def preparar_datos(sexo, predictores):
    """Filtra por sexo, separa X/y, ajusta imputación SOLO en train."""
    tr = df_train[df_train['Sex'] == sexo]
    te = df_test[df_test['Sex'] == sexo]
    # quitar 'Sex' de predictores (constante dentro del estrato)
    preds = [c for c in predictores if c != 'Sex']
    Xtr_raw, ytr = tr[preds].copy(), tr['SM_FINAL'].values
    Xte_raw, yte = te[preds].copy(), te['SM_FINAL'].values
    groups = tr['FOLIO'].values

    binarias, continuas = split_bin_cont(df_train, preds)
    # imputación ajustada en train
    imp_cont = SimpleImputer(strategy='median')
    imp_bin  = SimpleImputer(strategy='most_frequent')
    Xtr = Xtr_raw.copy(); Xte = Xte_raw.copy()
    if continuas:
        Xtr[continuas] = imp_cont.fit_transform(Xtr_raw[continuas])
        Xte[continuas] = imp_cont.transform(Xte_raw[continuas])
    if binarias:
        Xtr[binarias] = imp_bin.fit_transform(Xtr_raw[binarias])
        Xte[binarias] = imp_bin.transform(Xte_raw[binarias])
    return Xtr.values, ytr, Xte.values, yte, groups, preds

def calc_metricas(y_true, y_pred, y_prob):
    """Todas las métricas para un conjunto."""
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    spec = tn / (tn + fp) if (tn+fp) > 0 else 0
    ppv  = tp / (tp + fp) if (tp+fp) > 0 else 0
    return {
        'auc_roc': roc_auc_score(y_true, y_prob),
        'auc_pr':  average_precision_score(y_true, y_prob),
        'bal_acc': balanced_accuracy_score(y_true, y_pred),
        'sens':    recall_score(y_true, y_pred),
        'spec':    spec,
        'ppv':     ppv,
        'f1':      f1_score(y_true, y_pred),
    }

def cv_metricas(modelo_fn, X, y, groups, n_splits=10):
    """CV agrupada por FOLIO; devuelve media±DE de cada métrica."""
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True,
                                random_state=RANDOM_STATE)
    acum = {k: [] for k in ['auc_roc','auc_pr','bal_acc','sens','spec','ppv','f1']}
    for tr_idx, va_idx in sgkf.split(X, y, groups):
        m = modelo_fn()
        m.fit(X[tr_idx], y[tr_idx])
        prob = m.predict_proba(X[va_idx])[:, 1]
        pred = (prob >= 0.5).astype(int)
        mm = calc_metricas(y[va_idx], pred, prob)
        for k in acum: acum[k].append(mm[k])
    return {k: (np.mean(v), np.std(v)) for k, v in acum.items()}

def bootstrap_ic(y_true, y_prob, y_pred, n=1000):
    """IC 95% por bootstrap en test."""
    rng = np.random.RandomState(RANDOM_STATE)
    acum = {k: [] for k in ['auc_roc','auc_pr','bal_acc','sens','spec','ppv','f1']}
    idx = np.arange(len(y_true))
    for _ in range(n):
        s = rng.choice(idx, len(idx), replace=True)
        if len(np.unique(y_true[s])) < 2: continue
        mm = calc_metricas(y_true[s], y_pred[s], y_prob[s])
        for k in acum: acum[k].append(mm[k])
    return {k: (np.percentile(v,2.5), np.percentile(v,97.5)) for k,v in acum.items()}

print('✓ Funciones definidas')


# ------------------------------------------------------------
# construir_modelo: colocada aquí (funciones auxiliares) para que
# las celdas de evaluación y sensibilidad funcionen SIN reentrenar.
# Requiere RANDOM_STATE, xgb y lgb, definidos en la celda de setup.
# ------------------------------------------------------------
def construir_modelo(algo, best_params, y):
    if algo == 'logreg':
        return LogisticRegression(**best_params, class_weight='balanced',
                                  max_iter=2000, random_state=RANDOM_STATE)
    elif algo == 'rf':
        return RandomForestClassifier(**best_params, class_weight='balanced',
                                      random_state=RANDOM_STATE, n_jobs=-1)
    elif algo == 'xgb':
        spw = (y == 0).sum() / max((y == 1).sum(), 1)
        return xgb.XGBClassifier(**best_params, scale_pos_weight=spw,
                                 eval_metric='logloss', random_state=RANDOM_STATE, n_jobs=-1)
    else:
        spw = (y == 0).sum() / max((y == 1).sum(), 1)
        return lgb.LGBMClassifier(**best_params, scale_pos_weight=spw,
                                  random_state=RANDOM_STATE, n_jobs=-1, verbose=-1)


EJECUTAR HASTA AQUÍ (EL SIGUIENTE Y YA)

In [ ]:
# RECARGA RÁPIDA — modelos ya entrenados (NO reentrenar)
import pickle
with open(CKPT_PATH + 'mejores_modelos.pkl', 'rb') as f:
    saved = pickle.load(f)
resultados = saved['resultados']
mejores = saved['mejores']
tabla = saved['tabla']
print('✓ Modelos recargados desde checkpoint')
print('  Modelos finales:', mejores)

In [ ]:
# ============================================================
# CELDA 4: Optimización Optuna + entrenamiento
# 4 algoritmos × 2 sexos × 2 configuraciones = 16 modelos
# Objetivo: balanced accuracy en CV agrupada por FOLIO
# ============================================================
N_TRIALS = 50

def objetivo(trial, algo, X, y, groups):
    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    if algo == 'logreg':
        params = {'C': trial.suggest_float('C', 1e-3, 1e2, log=True),
                  'class_weight': 'balanced', 'max_iter': 2000,
                  'random_state': RANDOM_STATE}
        make = lambda: LogisticRegression(**params)
    elif algo == 'rf':
        params = {'n_estimators': trial.suggest_int('n_estimators', 100, 500),
                  'max_depth': trial.suggest_int('max_depth', 3, 20),
                  'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
                  'class_weight': 'balanced', 'random_state': RANDOM_STATE, 'n_jobs': -1}
        make = lambda: RandomForestClassifier(**params)
    elif algo == 'xgb':
        spw = (y == 0).sum() / max((y == 1).sum(), 1)
        params = {'n_estimators': trial.suggest_int('n_estimators', 100, 500),
                  'max_depth': trial.suggest_int('max_depth', 2, 10),
                  'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
                  'subsample': trial.suggest_float('subsample', 0.6, 1.0),
                  'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
                  'scale_pos_weight': spw, 'eval_metric': 'logloss',
                  'random_state': RANDOM_STATE, 'n_jobs': -1}
        make = lambda: xgb.XGBClassifier(**params)
    else:  # lgbm
        spw = (y == 0).sum() / max((y == 1).sum(), 1)
        params = {'n_estimators': trial.suggest_int('n_estimators', 100, 500),
                  'max_depth': trial.suggest_int('max_depth', 2, 12),
                  'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
                  'num_leaves': trial.suggest_int('num_leaves', 15, 127),
                  'subsample': trial.suggest_float('subsample', 0.6, 1.0),
                  'scale_pos_weight': spw, 'random_state': RANDOM_STATE,
                  'n_jobs': -1, 'verbose': -1}
        make = lambda: lgb.LGBMClassifier(**params)

    scores = []
    for tr_idx, va_idx in sgkf.split(X, y, groups):
        m = make(); m.fit(X[tr_idx], y[tr_idx])
        pred = (m.predict_proba(X[va_idx])[:, 1] >= 0.5).astype(int)
        scores.append(balanced_accuracy_score(y[va_idx], pred))
    return np.mean(scores)

def construir_modelo(algo, best_params, y):
    if algo == 'logreg':
        return LogisticRegression(**best_params, class_weight='balanced',
                                  max_iter=2000, random_state=RANDOM_STATE)
    elif algo == 'rf':
        return RandomForestClassifier(**best_params, class_weight='balanced',
                                      random_state=RANDOM_STATE, n_jobs=-1)
    elif algo == 'xgb':
        spw = (y == 0).sum() / max((y == 1).sum(), 1)
        return xgb.XGBClassifier(**best_params, scale_pos_weight=spw,
                                 eval_metric='logloss', random_state=RANDOM_STATE, n_jobs=-1)
    else:
        spw = (y == 0).sum() / max((y == 1).sum(), 1)
        return lgb.LGBMClassifier(**best_params, scale_pos_weight=spw,
                                  random_state=RANDOM_STATE, n_jobs=-1, verbose=-1)

ALGOS = ['logreg', 'rf', 'xgb', 'lgbm']
CONFIGS = {'A': predictores_A, 'B': predictores_B}
SEXOS = {0: 'Female', 1: 'Male'}

resultados = {}
for sexo, sexo_nombre in SEXOS.items():
    for cfg_nombre, preds in CONFIGS.items():
        Xtr, ytr, Xte, yte, groups, preds_usados = preparar_datos(sexo, preds)
        for algo in ALGOS:
            key = f'{sexo_nombre}_{cfg_nombre}_{algo}'
            print(f'Optimizando {key} ...', end=' ')
            study = optuna.create_study(direction='maximize')
            study.optimize(lambda t: objetivo(t, algo, Xtr, ytr, groups),
                           n_trials=N_TRIALS, show_progress_bar=False)
            modelo = construir_modelo(algo, study.best_params, ytr)
            modelo.fit(Xtr, ytr)
            resultados[key] = {
                'modelo': modelo, 'algo': algo, 'sexo': sexo,
                'cfg': cfg_nombre, 'preds': preds_usados,
                'best_params': study.best_params, 'best_cv_balacc': study.best_value,
                'Xtr': Xtr, 'ytr': ytr, 'Xte': Xte, 'yte': yte, 'groups': groups
            }
            print(f'CV bal_acc = {study.best_value:.3f}')

print('\n✓ 16 modelos entrenados')
with open(CKPT_PATH + 'modelos_entrenados.pkl', 'wb') as f:
    pickle.dump(resultados, f)
print('✓ Guardado: checkpoints/modelos_entrenados.pkl')

In [ ]:
# ============================================================
# CELDA 5: Evaluación (CV media±DE + test + bootstrap IC) y selección
# ============================================================
filas = []
for key, r in resultados.items():
    Xtr, ytr = r['Xtr'], r['ytr']
    Xte, yte = r['Xte'], r['yte']
    groups, algo = r['groups'], r['algo']

    cv = cv_metricas(lambda: construir_modelo(algo, r['best_params'], ytr),
                     Xtr, ytr, groups, n_splits=10)
    prob_te = r['modelo'].predict_proba(Xte)[:, 1]
    pred_te = (prob_te >= 0.5).astype(int)
    test = calc_metricas(yte, pred_te, prob_te)
    ic = bootstrap_ic(yte, prob_te, pred_te)

    sens_perf = test['sens'] >= 0.999 or test['spec'] >= 0.999
    filas.append({
        'modelo': key, 'sexo': r['sexo'], 'cfg': r['cfg'], 'algo': algo,
        'CV_balacc': f"{cv['bal_acc'][0]:.3f}±{cv['bal_acc'][1]:.3f}",
        'AUC_ROC': f"{test['auc_roc']:.3f} [{ic['auc_roc'][0]:.3f}-{ic['auc_roc'][1]:.3f}]",
        'AUC_PR':  f"{test['auc_pr']:.3f} [{ic['auc_pr'][0]:.3f}-{ic['auc_pr'][1]:.3f}]",
        'BalAcc':  f"{test['bal_acc']:.3f}",
        'Sens':    f"{test['sens']:.3f}",
        'Spec':    f"{test['spec']:.3f}",
        'PPV':     f"{test['ppv']:.3f}",
        'F1':      f"{test['f1']:.3f}",
        'excluir': sens_perf,
        '_auc': test['auc_roc'], '_balacc': test['bal_acc']
    })

tabla = pd.DataFrame(filas)
pd.set_option('display.max_columns', None, 'display.width', 200)
print("=== TODOS LOS MODELOS ===")
print(tabla[['modelo','CV_balacc','AUC_ROC','AUC_PR','BalAcc','Sens','Spec','PPV','F1','excluir']].to_string(index=False))

# Selección: mejor por sexo×config entre los NO excluidos
print("\n=== MEJORES MODELOS SELECCIONADOS ===")
mejores = {}
for sexo in ['Female','Male']:
    for cfg in ['A','B']:
        sub = tabla[(tabla.sexo==sexo)&(tabla.cfg==cfg)&(~tabla.excluir)]
        if len(sub)==0:
            print(f"  {sexo} {cfg}: ⚠ todos excluidos por Sens/Spec perfecta")
            continue
        best = sub.sort_values(['_balacc','_auc'], ascending=False).iloc[0]
        mejores[f'{sexo}_{cfg}'] = best['modelo']
        print(f"  {sexo} {cfg}: {best['algo'].upper()} | BalAcc={best['BalAcc']} | AUC={best['AUC_ROC']}")

with open(CKPT_PATH + 'mejores_modelos.pkl', 'wb') as f:
    pickle.dump({'tabla': tabla, 'mejores': mejores, 'resultados': resultados}, f)
print('\n✓ Guardado: checkpoints/mejores_modelos.pkl')

In [ ]:
with open(CKPT_PATH + 'mejores_modelos.pkl', 'rb') as f:
    saved = pickle.load(f)
tabla = saved['tabla']
resultados = saved['resultados']

# Mapear código numérico a nombre
tabla['sexo_nom'] = tabla['sexo'].map({0: 'Female', 1: 'Male'})

print(f"Excluidos por Sens/Spec perfecta: {tabla['excluir'].sum()} de {len(tabla)}\n")
print("=== MEJORES MODELOS SELECCIONADOS ===")
mejores = {}
for sexo in ['Female', 'Male']:
    for cfg in ['A', 'B']:
        sub = tabla[(tabla.sexo_nom == sexo) & (tabla.cfg == cfg) & (~tabla.excluir)]
        if len(sub) == 0:
            print(f"  {sexo} {cfg}: ⚠ sin candidatos")
            continue
        best = sub.sort_values(['_balacc', '_auc'], ascending=False).iloc[0]
        mejores[f'{sexo}_{cfg}'] = best['modelo']
        print(f"  {sexo} {cfg}: {best['algo'].upper():8s} | "
              f"BalAcc={best['BalAcc']} | AUC={best['AUC_ROC']} | "
              f"AUC-PR={best['AUC_PR']} | PPV={best['PPV']} | Sens={best['Sens']}")

with open(CKPT_PATH + 'mejores_modelos.pkl', 'wb') as f:
    pickle.dump({'tabla': tabla, 'mejores': mejores, 'resultados': resultados}, f)
print('\n✓ Selección corregida y guardada')

In [ ]:
import pickle
with open(CKPT_PATH + 'mejores_modelos.pkl', 'rb') as f:
    saved = pickle.load(f)
tabla = saved['tabla']
mejores = saved['mejores']

# Mostrar tabla completa de los 16 modelos con TODAS las métricas
cols_mostrar = ['modelo','CV_balacc','AUC_ROC','AUC_PR','BalAcc',
                'Sens','Spec','PPV','F1']
pd.set_option('display.max_columns', None, 'display.width', 220)
print("=== TODOS LOS MODELOS (incluye especificidad) ===")
print(tabla[cols_mostrar].to_string(index=False))

# Resaltar solo los 4 seleccionados con todas sus métricas
print("\n=== MODELOS FINALES SELECCIONADOS (todas las métricas) ===")
sel = tabla[tabla['modelo'].isin(mejores.values())]
for _, r in sel.iterrows():
    print(f"\n{r['modelo']}")
    print(f"  AUC-ROC: {r['AUC_ROC']}")
    print(f"  AUC-PR : {r['AUC_PR']}")
    print(f"  Balanced Accuracy: {r['BalAcc']}")
    print(f"  Sensibilidad: {r['Sens']}  |  Especificidad: {r['Spec']}")
    print(f"  PPV: {r['PPV']}  |  F1: {r['F1']}")
    print(f"  CV bal_acc (media±DE): {r['CV_balacc']}")

In [ ]:
# ============================================================
# CELL 6: Calibration (Brier + reliability curves + isotonic)
# ============================================================
import matplotlib.pyplot as plt
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import brier_score_loss
from sklearn.base import clone

with open(CKPT_PATH + 'mejores_modelos.pkl', 'rb') as f:
    saved = pickle.load(f)
mejores = saved['mejores']
resultados = saved['resultados']

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
calib_info = {}

orden = [('Female_A', 'Female Config A'), ('Female_B', 'Female Config B'),
         ('Male_A',   'Male Config A'),   ('Male_B',   'Male Config B')]

for ax, (key, titulo) in zip(axes.ravel(), orden):
    modelo_key = mejores[key]
    r = resultados[modelo_key]
    Xtr, ytr, Xte, yte = r['Xtr'], r['ytr'], r['Xte'], r['yte']

    # Uncalibrated model
    prob_uncal  = r['modelo'].predict_proba(Xte)[:, 1]
    brier_uncal = brier_score_loss(yte, prob_uncal)

    # Isotonic calibration (CalibratedClassifierCV, 5-fold on train)
    base = clone(r['modelo'])
    cal  = CalibratedClassifierCV(base, method='isotonic', cv=5)
    cal.fit(Xtr, ytr)
    prob_cal  = cal.predict_proba(Xte)[:, 1]
    brier_cal = brier_score_loss(yte, prob_cal)

    # Reliability curves
    ft_u, mp_u = calibration_curve(yte, prob_uncal, n_bins=10, strategy='quantile')
    ft_c, mp_c = calibration_curve(yte, prob_cal,   n_bins=10, strategy='quantile')

    ax.plot([0, 1], [0, 1], '--', color='gray',  label='Perfect calibration')
    ax.plot(mp_u, ft_u, 'o-', color='red',  label=f'Uncalibrated (Brier={brier_uncal:.3f})')
    ax.plot(mp_c, ft_c, 's-', color='blue', label=f'Isotonic (Brier={brier_cal:.3f})')
    ax.set_title(f'{titulo} — {r["algo"].upper()}')
    ax.set_xlabel('Mean predicted probability')
    ax.set_ylabel('Observed fraction of positives')
    ax.legend(fontsize=8)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)

    calib_info[key] = {
        'modelo':      modelo_key,
        'algo':        r['algo'],
        'brier_uncal': brier_uncal,
        'brier_cal':   brier_cal,
        'rango_uncal': (prob_uncal.min(), prob_uncal.max()),
        'rango_cal':   (prob_cal.min(),   prob_cal.max()),
        'calibrador':  cal
    }

plt.tight_layout()
OUT_PATH = BASE_PATH + 'outputs/'
os.makedirs(OUT_PATH, exist_ok=True)
plt.savefig(OUT_PATH + 'calibration_curves_FINAL.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n=== CALIBRATION SUMMARY ===")
for key, info in calib_info.items():
    delta = info['brier_uncal'] - info['brier_cal']
    print(f"\n{key} ({info['algo'].upper()}):")
    print(f"  Brier uncalibrated: {info['brier_uncal']:.4f} | isotonic: {info['brier_cal']:.4f} "
          f"({'improved' if delta > 0 else 'worsened'} {abs(delta):.4f})")
    print(f"  Prob. range uncalibrated: [{info['rango_uncal'][0]:.2f}, {info['rango_uncal'][1]:.2f}]")
    print(f"  Prob. range isotonic:     [{info['rango_cal'][0]:.2f},  {info['rango_cal'][1]:.2f}]")

with open(CKPT_PATH + 'calibracion.pkl', 'wb') as f:
    pickle.dump(calib_info, f)
print('\n✓ Saved: calibracion.pkl and outputs/calibration_curves_FINAL.png')

In [ ]:
# ============================================================
# CELDA 7: SHAP + importancia nativa comparadas
# Se explica el MODELO BASE (sin wrapper de calibración).
# ============================================================
import shap
import matplotlib.pyplot as plt
import numpy as np

with open(CKPT_PATH + 'mejores_modelos.pkl', 'rb') as f:
    saved = pickle.load(f)
mejores = saved['mejores']
resultados = saved['resultados']

orden = [('Female_A','Female Config A'), ('Female_B','Female Config B'),
         ('Male_A','Male Config A'), ('Male_B','Male Config B')]

shap_resultados = {}

for key, titulo in orden:
    modelo_key = mejores[key]
    r = resultados[modelo_key]
    modelo = r['modelo']          # modelo base (XGB o LGBM), no calibrado
    Xte = r['Xte']
    preds = r['preds']
    Xte_df = pd.DataFrame(Xte, columns=preds)

    print(f'\n{"="*60}\n{titulo} — {r["algo"].upper()}\n{"="*60}')

    # --- SHAP (TreeExplainer, exacto para XGB y LGBM) ---
    explainer = shap.TreeExplainer(modelo)
    shap_vals = explainer.shap_values(Xte_df)
    # LightGBM binario devuelve lista [clase0, clase1]; tomar clase positiva
    if isinstance(shap_vals, list):
        shap_vals = shap_vals[1]
    # algunos XGB devuelven 2D directo; asegurar forma
    if shap_vals.ndim == 3:
        shap_vals = shap_vals[:, :, 1]

    # Importancia SHAP global = media del valor absoluto
    shap_imp = np.abs(shap_vals).mean(axis=0)
    shap_rank = pd.Series(shap_imp, index=preds).sort_values(ascending=False)

    # --- Importancia nativa del modelo ---
    if r['algo'] == 'xgb':
        nat = modelo.feature_importances_   # gain-based por defecto en sklearn API
    else:  # lgbm
        nat = modelo.feature_importances_
    nat_rank = pd.Series(nat, index=preds).sort_values(ascending=False)
    # normalizar para comparar
    nat_rank_norm = nat_rank / nat_rank.sum()

    # --- Tabla comparativa top 15 ---
    print(f"\n{'Rank':<5}{'SHAP (top 15)':<25}{'Importancia nativa (top 15)':<25}")
    for i in range(15):
        s_var = shap_rank.index[i] if i < len(shap_rank) else ''
        s_val = shap_rank.iloc[i] if i < len(shap_rank) else 0
        n_var = nat_rank.index[i] if i < len(nat_rank) else ''
        n_val = nat_rank_norm.iloc[i] if i < len(nat_rank_norm) else 0
        print(f"{i+1:<5}{s_var:<18}{s_val:6.4f}   {n_var:<18}{n_val:6.4f}")

    # Coincidencia de top 10
    top10_shap = set(shap_rank.index[:10])
    top10_nat = set(nat_rank.index[:10])
    overlap = len(top10_shap & top10_nat)
    print(f"\n  Coincidencia top-10 SHAP vs nativa: {overlap}/10 variables")

    # --- Guardar SHAP summary plot ---
    plt.figure()
    shap.summary_plot(shap_vals, Xte_df, max_display=15, show=False)
    plt.title(f'{titulo} — {r["algo"].upper()}')
    plt.tight_layout()
    fname = OUT_PATH + f'shap_{key}.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.close()

    shap_resultados[key] = {
        'shap_rank': shap_rank, 'nat_rank': nat_rank,
        'shap_vals': shap_vals, 'overlap_top10': overlap,
        'modelo_key': modelo_key
    }

with open(CKPT_PATH + 'shap_resultados.pkl', 'wb') as f:
    pickle.dump(shap_resultados, f)
print(f'\n✓ Guardado: shap_resultados.pkl y 4 plots SHAP en outputs/')

In [ ]:
# ============================================================
# CELDA 8: Análisis de sensibilidad — modelos SIN componentes ATP III continuos
# Excluye WC, SBP, DBP y reentrena con los mismos hiperparámetros/algoritmo.
# Compara contra el rendimiento de los modelos completos.
# ============================================================
import numpy as np

with open(CKPT_PATH + 'mejores_modelos.pkl', 'rb') as f:
    saved = pickle.load(f)
mejores = saved['mejores']
resultados = saved['resultados']
tabla = saved['tabla']

# Componentes ATP III que entran como continuos
ATP3_CONTINUOS = ['WC', 'SBP', 'DBP']

orden = [('Female_A','Female Config A'), ('Female_B','Female Config B'),
         ('Male_A','Male Config A'), ('Male_B','Male Config B')]

print("=== ANÁLISIS DE SENSIBILIDAD ===")
print("Comparación: modelo COMPLETO vs SIN (WC, SBP, DBP)\n")

sens_resultados = {}
for key, titulo in orden:
    modelo_key = mejores[key]
    r = resultados[modelo_key]
    algo = r['algo']
    preds = r['preds']

    # Índices de columnas a conservar (quitar ATP3_CONTINUOS)
    keep_idx = [i for i, c in enumerate(preds) if c not in ATP3_CONTINUOS]
    preds_reducidos = [preds[i] for i in keep_idx]
    Xtr_red = r['Xtr'][:, keep_idx]
    Xte_red = r['Xte'][:, keep_idx]
    ytr, yte = r['ytr'], r['yte']

    # Reentrenar con mismos hiperparámetros y algoritmo
    modelo_red = construir_modelo(algo, r['best_params'], ytr)
    modelo_red.fit(Xtr_red, ytr)
    prob = modelo_red.predict_proba(Xte_red)[:, 1]
    pred = (prob >= 0.5).astype(int)
    m_red = calc_metricas(yte, pred, prob)

    # Métricas del modelo completo (de la tabla)
    fila_full = tabla[tabla['modelo'] == modelo_key].iloc[0]
    auc_full = fila_full['_auc']
    balacc_full = fila_full['_balacc']

    print(f"{titulo} ({algo.upper()}):")
    print(f"  COMPLETO:           AUC={auc_full:.3f}  BalAcc={balacc_full:.3f}")
    print(f"  SIN WC/SBP/DBP:     AUC={m_red['auc_roc']:.3f}  BalAcc={m_red['bal_acc']:.3f}  "
          f"Sens={m_red['sens']:.3f}  Spec={m_red['spec']:.3f}  PPV={m_red['ppv']:.3f}")
    print(f"  Caída:              ΔAUC={auc_full-m_red['auc_roc']:+.3f}  "
          f"ΔBalAcc={balacc_full-m_red['bal_acc']:+.3f}\n")

    sens_resultados[key] = {
        'auc_full': auc_full, 'auc_red': m_red['auc_roc'],
        'balacc_full': balacc_full, 'balacc_red': m_red['bal_acc'],
        'metricas_red': m_red, 'preds_reducidos': preds_reducidos
    }

with open(CKPT_PATH + 'sensibilidad.pkl', 'wb') as f:
    pickle.dump(sens_resultados, f)
print('✓ Guardado: sensibilidad.pkl')

In [ ]:
import pickle
with open(CKPT_PATH + 'split_data.pkl', 'rb') as f:
    d = pickle.load(f)
df_train, df_test = d['df_train'], d['df_test']

# Reunir todo el dataset (train+test) para tener trayectorias completas
df_full = pd.concat([df_train, df_test], ignore_index=True)

# Incidentes
inc = df_full[df_full['OUTCOME'] == 'INCIDENT']
print(f"Observaciones de incidentes: {len(inc)}")
print(f"Participantes incidentes únicos: {inc['FOLIO'].nunique()}")
print(f"  Mujeres: {inc[inc.Sex==0]['FOLIO'].nunique()} | Hombres: {inc[inc.Sex==1]['FOLIO'].nunique()}")

# Visitas por incidente
visitas_por_inc = inc.groupby('FOLIO').size()
print(f"\nDistribución de número de visitas por incidente:")
print(visitas_por_inc.value_counts().sort_index().to_string())

# Confirmar que VISITA permite ordenar
print(f"\nValores de VISITA: {sorted(df_full['VISITA'].unique())}")

# Confirmar que SM_FINAL marca la transición (0 en baseline, 1 después)
ej = inc[inc['FOLIO']==inc['FOLIO'].iloc[0]].sort_values('VISITA')
print(f"\nEjemplo de trayectoria (FOLIO {ej['FOLIO'].iloc[0]}):")
print(ej[['FOLIO','VISITA','SM_FINAL','WC','athero_index']].to_string(index=False))

In [ ]:
# ============================================================
# CELDA 9: Validación direccional contra trayectorias de incidentes
# Para cada incidente: compara el cambio observado (baseline -> visita SM)
# en cada variable simulable, contra la dirección de cambio de riesgo
# que predice el modelo al aplicar ese mismo cambio al perfil baseline.
# ============================================================
import numpy as np

with open(CKPT_PATH + 'split_data.pkl', 'rb') as f:
    d = pickle.load(f)
with open(CKPT_PATH + 'mejores_modelos.pkl', 'rb') as f:
    saved = pickle.load(f)
with open(CKPT_PATH + 'calibracion.pkl', 'rb') as f:
    calib_info = pickle.load(f)

df_full = pd.concat([d['df_train'], d['df_test']], ignore_index=True)
mejores = saved['mejores']
resultados = saved['resultados']

# Sets simulables
SIM_A = ['WC', 'BMI', 'Weight', 'SBP', 'DBP', 'mets_total', 'sit_week_min']
SIM_B = SIM_A + ['athero_index', 'cholesterol', 'uric_acid', 'ldl']

def obtener_trayectorias(sexo):
    """Para cada incidente del sexo dado, devuelve (baseline_row, sm_row)."""
    inc = df_full[(df_full['OUTCOME']=='INCIDENT') & (df_full['Sex']==sexo)]
    trayectorias = []
    for folio, g in inc.groupby('FOLIO'):
        g = g.sort_values('VISITA')
        baseline = g.iloc[0]                      # primera visita (SM=0)
        sm_rows = g[g['SM_FINAL']==1]
        if len(sm_rows)==0: continue
        sm_visit = sm_rows.iloc[0]                # primera visita con SM=1
        trayectorias.append((baseline, sm_visit))
    return trayectorias

def validar_direccional(key, sim_vars):
    """Concordancia direccional por variable para un modelo."""
    modelo_key = mejores[key]
    r = resultados[modelo_key]
    sexo = r['sexo']
    preds = r['preds']
    modelo = r['modelo']
    # imputador implícito: usamos medianas del train para llenar NaN del perfil
    Xtr_df = pd.DataFrame(r['Xtr'], columns=preds)
    medianas = Xtr_df.median()

    trayectorias = obtener_trayectorias(sexo)
    conteo = {v: {'conc':0, 'disc':0} for v in sim_vars if v in preds}

    for baseline, sm_visit in trayectorias:
        # perfil baseline en el espacio de predictores
        perfil = pd.Series({c: baseline[c] if c in baseline and not pd.isna(baseline[c])
                            else medianas[c] for c in preds})
        riesgo_base = modelo.predict_proba(perfil.values.reshape(1,-1))[0,1]

        for v in conteo:
            if pd.isna(baseline[v]) or pd.isna(sm_visit[v]):
                continue
            delta_obs = sm_visit[v] - baseline[v]
            if delta_obs == 0:
                continue  # sin dirección evaluable
            # aplicar el cambio observado al perfil baseline
            perfil_mod = perfil.copy()
            perfil_mod[v] = sm_visit[v]
            riesgo_mod = modelo.predict_proba(perfil_mod.values.reshape(1,-1))[0,1]
            delta_riesgo = riesgo_mod - riesgo_base
            # concordante si signo del cambio de riesgo coincide con
            # la transición a enfermedad (incidente => debería SUBIR riesgo)
            if delta_riesgo == 0:
                continue
            # como el incidente desarrolló SM, un cambio "hacia enfermedad"
            # debería aumentar el riesgo. Concordancia = signo(delta_riesgo)
            # coincide con signo esperado dado el signo del delta observado.
            # Criterio del paper: signo(delta_obs) vs signo(delta_riesgo)
            if np.sign(delta_obs) == np.sign(delta_riesgo):
                conteo[v]['conc'] += 1
            else:
                conteo[v]['disc'] += 1
    return conteo

# Ejecutar para los 4 modelos
sim_sets = {'Female_A': SIM_A, 'Female_B': SIM_B,
            'Male_A': SIM_A, 'Male_B': SIM_B}

global_conc, global_disc = 0, 0
por_variable = {}
for key, sim in sim_sets.items():
    conteo = validar_direccional(key, sim)
    print(f"\n=== {key} ===")
    print(f"{'Variable':<16}{'Conc':>6}{'Disc':>6}{'Concordancia %':>16}")
    for v, c in conteo.items():
        tot = c['conc'] + c['disc']
        if tot == 0: continue
        pct = 100*c['conc']/tot
        print(f"{v:<16}{c['conc']:>6}{c['disc']:>6}{pct:>15.1f}%")
        global_conc += c['conc']; global_disc += c['disc']
        por_variable.setdefault(v, {'conc':0,'disc':0})
        por_variable[v]['conc'] += c['conc']
        por_variable[v]['disc'] += c['disc']

print(f"\n{'='*50}")
print("=== CONCORDANCIA POR VARIABLE (agregada) ===")
print(f"{'Variable':<16}{'Conc':>6}{'Disc':>6}{'Concordancia %':>16}")
for v in sorted(por_variable, key=lambda x: -por_variable[x]['conc']/max(por_variable[x]['conc']+por_variable[x]['disc'],1)):
    c = por_variable[v]; tot = c['conc']+c['disc']
    print(f"{v:<16}{c['conc']:>6}{c['disc']:>6}{100*c['conc']/tot:>15.1f}%")

total = global_conc + global_disc
print(f"\n{'='*50}")
print(f"CONCORDANCIA DIRECCIONAL GLOBAL: {100*global_conc/total:.1f}% ({global_conc}/{total})")

with open(CKPT_PATH + 'validacion_direccional.pkl', 'wb') as f:
    pickle.dump({'por_variable': por_variable, 'global':(global_conc,total)}, f)
print('✓ Guardado: validacion_direccional.pkl')

In [ ]:
with open(CKPT_PATH + 'validacion_direccional.pkl', 'rb') as f:
    vd = pickle.load(f)
pv = vd['por_variable']

# Set simulable FINAL: excluir LDL (redundante/invertido) y
# las variables de IPAQ que no muestran consistencia direccional
EXCLUIR = ['ldl', 'sit_week_min', 'mets_total']

print("=== ESCENARIOS DE SET SIMULABLE ===\n")

def global_con(excluidos):
    c = d_ = 0
    for v, ct in pv.items():
        if v in excluidos: continue
        c += ct['conc']; d_ += ct['disc']
    return c, c+d_

# Escenario 1: solo excluir LDL (como paper original)
c, t = global_con(['ldl'])
print(f"Excluyendo solo LDL:                {100*c/t:.1f}% ({c}/{t})")

# Escenario 2: excluir LDL + sedentarismo
c, t = global_con(['ldl','sit_week_min'])
print(f"Excluyendo LDL + sedentarismo:      {100*c/t:.1f}% ({c}/{t})")

# Escenario 3: excluir LDL + ambas IPAQ
c, t = global_con(['ldl','sit_week_min','mets_total'])
print(f"Excluyendo LDL + IPAQ (mets+sit):   {100*c/t:.1f}% ({c}/{t})")

print("\n=== CONCORDANCIA POR VARIABLE (set final, sin LDL/IPAQ) ===")
for v in sorted(pv, key=lambda x: -pv[x]['conc']/max(pv[x]['conc']+pv[x]['disc'],1)):
    if v in EXCLUIR: continue
    ct = pv[v]; tot = ct['conc']+ct['disc']
    print(f"  {v:<16}{100*ct['conc']/tot:>6.1f}%  ({ct['conc']}/{tot})")

In [ ]:
# ============================================================
# Tabla 2 — Características basales por grupo (etiqueta corregida)
# ============================================================
import pandas as pd, numpy as np

raw = pd.read_csv(BASE_PATH + 'dataset_oficial.csv')

# Reconstruir etiqueta y outcome
def atp3(row):
    if row['Sex'] == 1:
        cint = int(row['WC'] > 102); hdl = int(row['hdl'] < 40)
    else:
        cint = int(row['WC'] > 88);  hdl = int(row['hdl'] < 50)
    return cint + hdl + int(row['glucose'] >= 100) + int(row['triglycerides'] >= 150) + int((row['SBP'] >= 130) or (row['DBP'] >= 85))

raw['cnt'] = raw.apply(atp3, axis=1)
raw['SM_ok'] = (raw['cnt'] >= 3).astype(int)
raw = raw.sort_values(['FOLIO', 'VISITA'])

def oc(g):
    g = g.sort_values('VISITA')
    if g.iloc[0]['SM_ok'] == 1:
        return 'PREVALENT'
    elif g['SM_ok'].max() == 1:
        return 'INCIDENT'
    return 'CONTROL'

ocm = raw.groupby('FOLIO').apply(oc, include_groups=False)
raw['grp'] = raw['FOLIO'].map(ocm)

# Baseline por participante (primera visita)
base = raw.loc[raw.groupby('FOLIO')['VISITA'].idxmin()].copy()
base['grp'] = base['FOLIO'].map(ocm)

# --- Continuas ---
cont_vars = ['Age', 'Weight', 'Height', 'BMI', 'WC', 'SBP', 'DBP',
             'mets_total', 'sit_week_min', 'anxiety_trait']
print("=== CONTINUAS (media +/- DE) por grupo ===")
print(f"{'Variable':<16}{'Prevalent':>22}{'Incident':>22}{'Control':>22}")
for v in cont_vars:
    row = f"{v:<16}"
    for g in ['PREVALENT', 'INCIDENT', 'CONTROL']:
        sub = base[base.grp == g][v].dropna()
        cell = f"{sub.mean():.1f} +/- {sub.std():.1f}"
        row += f"{cell:>22}"
    print(row)

# --- Categoricas ---
print("\n=== CATEGORICAS (%) por grupo ===")
cat_vars = {'Women': ('Sex', 0), 'mom_diab': ('mom_diab', 1), 'dad_diab': ('dad_diab', 1),
            'mom_htn': ('mom_htn', 1), 'mom_obese': ('mom_obese', 1),
            'mom_dyslip': ('mom_dyslip', 1), 'dad_dyslip': ('dad_dyslip', 1)}
print(f"{'Variable':<16}{'Prevalent':>12}{'Incident':>12}{'Control':>12}")
for label, (col, val) in cat_vars.items():
    if col not in base.columns:
        print(f"{label:<16} (columna no encontrada: {col})")
        continue
    row = f"{label:<16}"
    for g in ['PREVALENT', 'INCIDENT', 'CONTROL']:
        sub = base[base.grp == g]
        row += f"{100*(sub[col]==val).mean():>12.1f}"
    print(row)

n_prev = int((base.grp == 'PREVALENT').sum())
n_inc  = int((base.grp == 'INCIDENT').sum())
n_ctrl = int((base.grp == 'CONTROL').sum())
print(f"\nN por grupo: PREVALENT={n_prev}, INCIDENT={n_inc}, CONTROL={n_ctrl}")

In [ ]:
import pickle
import pandas as pd
with open(CKPT_PATH + 'split_data.pkl', 'rb') as f:
    d = pickle.load(f)
df_train = d['df_train']
preds_B = [c for c in d['predictores_B'] if c != 'Sex']

# Solo continuas (más de 2 valores únicos)
continuas = [c for c in preds_B if df_train[c].dropna().nunique() > 2]
print(f"Predictores continuos: {len(continuas)}")

corr = df_train[continuas].corr().abs()
import numpy as np
pares = []
for i in range(len(continuas)):
    for j in range(i+1, len(continuas)):
        r = corr.iloc[i, j]
        if r > 0.80:
            pares.append((continuas[i], continuas[j], r))
pares.sort(key=lambda x: -x[2])
print(f"\nPares con |r| > 0.80: {len(pares)}")
for a, b, r in pares:
    print(f"  {a:<16} {b:<16} {r:.3f}")

In [ ]:
# ============================================================
# Figures 2 & 3 — SHAP summary plots (female / male)
# ============================================================
import shap
import pickle
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

with open(CKPT_PATH + 'mejores_modelos.pkl', 'rb') as f:
    saved = pickle.load(f)
mejores   = saved['mejores']
resultados = saved['resultados']


def shap_panel(keys, titulos, fname, suptitle):
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    for ax, key, tit in zip(axes, keys, titulos):
        r      = resultados[mejores[key]]
        Xte_df = pd.DataFrame(r['Xte'], columns=r['preds'])
        expl   = shap.TreeExplainer(r['modelo'])
        sv     = expl.shap_values(Xte_df)

        if isinstance(sv, list):  # LightGBM binary → take positive class
            sv = sv[1]
        if sv.ndim == 3:          # XGBoost 3-D output → take positive class
            sv = sv[:, :, 1]

        plt.sca(ax)
        shap.summary_plot(sv, Xte_df, max_display=15, show=False, plot_size=None)
        ax.set_title(tit, fontsize=12)

    plt.suptitle(suptitle, fontsize=14)
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.savefig(OUT_PATH + fname, dpi=300, bbox_inches='tight')
    plt.show()
    print(f'✓ Saved: {fname}')


# Figure 2 — Female
shap_panel(
    ['Female_A', 'Female_B'],
    ['Configuration A (XGBoost, without laboratory tests)',
     'Configuration B (LightGBM, with laboratory tests)'],
    'shap_female_comparison_EN.png',
    'SHAP Summary — Female Models'
)

# Figure 3 — Male
shap_panel(
    ['Male_A', 'Male_B'],
    ['Configuration A (LightGBM, without laboratory tests)',
     'Configuration B (XGBoost, with laboratory tests)'],
    'shap_male_comparison_EN.png',
    'SHAP Summary — Male Models'
)

In [ ]:
# ============================================================
# Figure 5 - Waist-circumference threshold curves: male + female (2 panels)
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
import pickle

with open(CKPT_PATH + 'mejores_modelos.pkl', 'rb') as f:
    saved = pickle.load(f)
mejores = saved['mejores']; resultados = saved['resultados']

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

config = [
    ('Male_A',   'Male (Configuration A)',   102, 'steelblue'),
    ('Female_A', 'Female (Configuration A)',  88, 'crimson'),
]

for ax, (key, titulo, atp3_cut, color) in zip(axes, config):
    r = resultados[mejores[key]]
    modelo = r['modelo']; preds = r['preds']
    Xte = r['Xte']; yte = r['yte']
    idx_wc = preds.index('WC')

    prob_te = modelo.predict_proba(Xte)[:, 1]
    cand = np.where((yte == 1) & (prob_te > 0.6))[0]
    pac = cand[np.argmax(Xte[cand, idx_wc])]
    perfil = Xte[pac].copy()
    wc_real = perfil[idx_wc]

    wc_vals = Xte[:, idx_wc]
    wc_grid = np.linspace(np.percentile(wc_vals, 5), np.percentile(wc_vals, 95), 200)
    riesgos = np.array([
        modelo.predict_proba(np.where(np.arange(len(perfil)) == idx_wc, w, perfil).reshape(1, -1))[0, 1]
        for w in wc_grid])
    pend = np.gradient(riesgos, wc_grid)
    umbral = wc_grid[np.argmax(pend)]


    ax.plot(wc_grid, riesgos * 100, color=color, lw=2.2, label='Predicted risk')
    ax.axvline(atp3_cut, color='green', ls=':', lw=2, label=f'ATP III cut-off ({atp3_cut} cm)')
    ax.axvline(umbral, color='orange', ls='--', lw=2, label=f'Model threshold (≈{umbral:.1f} cm)')
    ax.set_xlabel('Waist circumference (cm)')
    ax.set_ylabel('Predicted MetS risk (%)')
    ax.set_title(titulo, fontsize=12)
    ax.legend(fontsize=9); ax.set_ylim(0, 100); ax.grid(alpha=0.25)

plt.suptitle('Individual Risk Curves for Waist Circumference', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(OUT_PATH + 'fig_threshold_WC_both.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Guardado: outputs/fig_threshold_WC_both.png')

In [ ]:
# ============================================================
# Slope / risk-range summary for the WC threshold curves (Figure 5).
# Recomputed directly from the same curves shown in Figure 5, for both sexes.
# ============================================================
import numpy as np
for key in ['Male_A', 'Female_A']:
    r = resultados[mejores[key]]
    modelo = r['modelo']; preds = r['preds']; Xte = r['Xte']; yte = r['yte']
    idx_wc = preds.index('WC')
    prob_te = modelo.predict_proba(Xte)[:, 1]
    cand = np.where((yte == 1) & (prob_te > 0.6))[0]
    pac = cand[np.argmax(Xte[cand, idx_wc])]
    perfil = Xte[pac].copy()
    wc_vals = Xte[:, idx_wc]
    wc_grid = np.linspace(np.percentile(wc_vals, 5), np.percentile(wc_vals, 95), 200)
    riesgos = np.array([
        modelo.predict_proba(np.where(np.arange(len(perfil)) == idx_wc, w, perfil).reshape(1, -1))[0, 1]
        for w in wc_grid])
    pend = np.gradient(riesgos, wc_grid)
    imax = int(np.argmax(pend))
    print(f"--- {key} ---")
    print(f"  max slope = {pend.max():.3f}  at WC = {wc_grid[imax]:.1f} cm")
    print(f"  risk before/after threshold = {riesgos[max(imax-3,0)]*100:.0f}% -> {riesgos[min(imax+3,199)]*100:.0f}%")
    print(f"  min/max risk over range = {riesgos.min()*100:.0f}% / {riesgos.max()*100:.0f}%")


In [ ]:
import pickle
with open(CKPT_PATH + 'split_data.pkl', 'rb') as f:
    d = pickle.load(f)
preds_B = [c for c in d['predictores_B'] if c != 'Sex']
preds_A = [c for c in d['predictores_A'] if c != 'Sex']
solo_B = [c for c in preds_B if c not in preds_A]

print("=== PREDICTORES (Config A + B) ===")
for c in sorted(preds_B):
    marca = " [solo B]" if c in solo_B else ""
    print(f"  {c}{marca}")
print(f"\nTotal A: {len(preds_A)} | Total B: {len(preds_B)} | Solo B (lab): {len(solo_B)}")

In [ ]:
# ============================================================
# CELDA 12: Multifactorial scenario — both sexes (Config B)  [Figure 5]
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle, os

OUT_PATH = BASE_PATH + 'outputs/'
os.makedirs(OUT_PATH, exist_ok=True)

with open(CKPT_PATH + 'mejores_modelos.pkl', 'rb') as f:
    saved = pickle.load(f)
mejores = saved['mejores']
resultados = saved['resultados']

SIM_B_FINAL = ['WC', 'BMI', 'Weight', 'SBP', 'DBP', 'athero_index', 'cholesterol', 'uric_acid']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (key, titulo, color) in zip(axes, [
        ('Female_B','Women (Config B)','crimson'),
        ('Male_B','Men (Config B)','steelblue')]):
    modelo_key = mejores[key]
    r = resultados[modelo_key]
    modelo = r['modelo']; preds = r['preds']
    Xte = r['Xte']; yte = r['yte']
    Xtr_df = pd.DataFrame(r['Xtr'], columns=preds)

    prob_te = modelo.predict_proba(Xte)[:, 1]
    # patient with intermediate-to-high risk (~0.72)
    paciente_idx = np.argmin(np.abs(prob_te - 0.72))
    perfil = Xte[paciente_idx].copy()
    riesgo_base = prob_te[paciente_idx]

    sim_vars = [v for v in SIM_B_FINAL if v in preds]
    reducciones = {}
    for v in sim_vars:
        idx = preds.index(v)
        p5 = np.percentile(Xtr_df[v], 5)
        p_mod = perfil.copy(); p_mod[idx] = p5
        riesgo_mod = modelo.predict_proba(p_mod.reshape(1,-1))[0,1]
        reducciones[v] = (riesgo_base - riesgo_mod) * 100

    orden_v = sorted(reducciones, key=reducciones.get)
    vals = [reducciones[v] for v in orden_v]
    ax.barh(orden_v, vals, color=color, alpha=0.8)
    for i, val in enumerate(vals):
        ax.text(val, i, f' {val:+.1f}', va='center', fontsize=9)
    ax.set_xlabel('Risk reduction (percentage points)')
    ax.set_title(f'{titulo}\nBaseline risk {riesgo_base*100:.0f}%')
    ax.axvline(0, color='gray', lw=0.8)

plt.tight_layout()
plt.savefig(OUT_PATH + 'fig_multifactorial_both.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Saved: outputs/fig_multifactorial_both.png")

In [ ]:
# ============================================================
# Figure 4 - Worked patient-level example (paired SHAP + risk before/after)
# Reproduces figura_s363.png using the packaged simulator (metsim.py).
# Locates the exact documented patient (Female, Config B) by matching both
# endpoints of the paper: calibrated baseline ~72.1% and post-modification
# ~30.8%, under the two-lever change (WC -8.0 cm, atherogenic index -0.58).
# ============================================================
import os, sys, urllib.request, numpy as np

REPO = 'https://raw.githubusercontent.com/GuEsGG/mets-whatif-simulator/main/'
for fname in ['metsim.py', 'simulator_artifacts.pkl']:
    if not os.path.exists(fname):
        print(f'Downloading {fname} ...')
        urllib.request.urlretrieve(REPO + fname, fname)
sys.path.insert(0, os.getcwd())
from metsim import MetSimulator

sim = MetSimulator('simulator_artifacts.pkl')
key_fb, sexo_fb, cfg_fb = sim._key('female', True)
cal_fb = sim.A['calibrators'][key_fb]
pdlt   = sim.A['plausible_delta'][sexo_fb]

r_fb     = resultados[mejores['Female_B']]
preds_fb = r_fb['preds']; Xte_fb = r_fb['Xte']
MODS = {'WC': -8.0, 'athero_index': -0.58}   # deltas del paper

def base_mod(prof):
    x0, mp = sim._vector(key_fb, prof)
    b = cal_fb.predict_proba(x0.reshape(1, -1))[0, 1]
    x1 = x0.copy()
    for v, d in MODS.items():
        i = mp.index(v); lo, hi = pdlt.get(v, (-np.inf, np.inf))
        x1[i] = x0[i] + float(np.clip(d, lo, hi))
    return b, cal_fb.predict_proba(x1.reshape(1, -1))[0, 1]

# Paciente documentada: base ~0.721 y modificado ~0.308
scores = []
for j in range(len(Xte_fb)):
    prof = {f: float(Xte_fb[j, i]) for i, f in enumerate(preds_fb)}
    b, m = base_mod(prof)
    scores.append(abs(b - 0.721) + abs(m - 0.308))
pac     = int(np.argmin(scores))
profile = {f: float(Xte_fb[pac, i]) for i, f in enumerate(preds_fb)}

res = sim.simulate_patient(
    profile, sex='female', has_lab=True,
    modifications={'WC': profile['WC'] - 8.0,
                   'athero_index': profile['athero_index'] - 0.58})
print(f"Selected patient WC = {profile['WC']:.1f} cm")
print(f"Baseline calibrated risk: {res['baseline_risk']*100:.1f}%")
print(f"Modified calibrated risk: {res['modified_risk']*100:.1f}%")
sim.plot_simulation(res, save=OUT_PATH + 'figura_s363.png')
print('Saved: outputs/figura_s363.png')
